In [1]:
from machine_learning.load_data import importing_preprocessed_data

preprocessed_data = importing_preprocessed_data()

In [2]:
print('Dataset shape:', preprocessed_data.shape)

print('\nAll columns:')
for column_number, column_name in enumerate(preprocessed_data.columns):
    print(f'{column_number}: {column_name}')

Dataset shape: (500, 24)

All columns:
0: session_id
1: app_name
2: app_category
3: previous_app
4: timestamp
5: hour_of_day
6: day_of_week
7: is_weekend
8: session_duration_sec
9: session_duration_hms
10: cpu_usage_pct
11: ram_usage_mb
12: network_usage_mb
13: battery_drain_pct
14: is_anomaly
15: date
16: session_duration_min
17: day_part
18: session_duration_sec_outlier
19: cpu_usage_pct_outlier
20: ram_usage_mb_outlier
21: network_usage_mb_outlier
22: battery_drain_pct_outlier
23: previous_to_current_probability


In [3]:
from machine_learning.load_data import run_load_data

(preprocessed_data, app_name_prev, day_of_week, day_part, app_category) = run_load_data()

In [4]:
print('\nApp name encoding(text to number):')
print(dict(zip(app_name_prev.classes_, app_name_prev.transform(app_name_prev.classes_))))

print('\nApp category encoding(category to number):')
print(dict(zip(app_category.classes_, app_category.transform(app_category.classes_))))


App name encoding(text to number):
{'Chrome': np.int64(0), 'Discord': np.int64(1), 'Excel': np.int64(2), 'Figma': np.int64(3), 'File Explorer': np.int64(4), 'NONE': np.int64(5), 'Notepad': np.int64(6), 'Notion': np.int64(7), 'OBS Studio': np.int64(8), 'Photoshop': np.int64(9), 'Postman': np.int64(10), 'PyCharm': np.int64(11), 'Slack': np.int64(12), 'Spotify': np.int64(13), 'Steam': np.int64(14), 'Task Manager': np.int64(15), 'Terminal': np.int64(16), 'VLC': np.int64(17), 'VS Code': np.int64(18), 'Word': np.int64(19), 'Zoom': np.int64(20)}

App category encoding(category to number):
{'Browser': np.int64(0), 'Communication': np.int64(1), 'Design': np.int64(2), 'Development': np.int64(3), 'Entertainment': np.int64(4), 'Productivity': np.int64(5), 'System': np.int64(6)}


In [5]:
# features and targets
from machine_learning.features import run_features

(INPUT_FEATURES, feature_matrix, predicted_next_app, anomaly) = run_features()

In [6]:
print(f'\nTotal input features: {len(INPUT_FEATURES)} out of {len(preprocessed_data.columns)} columns')
print('\nFeatures: ', INPUT_FEATURES)


Total input features: 17 out of 29 columns

Features:  ['previous_app_label', 'app_category_label', 'hour_of_day', 'day_of_week_label', 'day_part_label', 'is_weekend', 'session_duration_min', 'cpu_usage_pct', 'ram_usage_mb', 'network_usage_mb', 'battery_drain_pct', 'session_duration_sec_outlier', 'cpu_usage_pct_outlier', 'ram_usage_mb_outlier', 'network_usage_mb_outlier', 'battery_drain_pct_outlier', 'previous_to_current_probability']


In [7]:
# split into training and testing
from machine_learning.split import run_split

(X_train, X_test, y_train_app, y_test_app, y_train_anomaly, y_test_anomaly) = run_split()

In [8]:
print(f'\nTraining rows: {len(X_train)}')
print(f'\nTest rows: {len(X_test)}')


Training rows: 400

Test rows: 100


In [9]:
print('Anomaly distribution in test set:')
print(y_test_anomaly.value_counts().to_string())

Anomaly distribution in test set:
is_anomaly
0    99
1     1


In [10]:
# train all models and see the differences
from machine_learning.train import run_train

(markov_transition_table, next_app_all_results, next_app_top_model, next_app_top_model_name, next_app_top_model_type, anomaly_all_results, anomaly_all_confusion_matrices, best_anomaly_model, best_anomaly_model_name) = run_train()

In [11]:
import pandas as pd

# this is for model 1: next app predictor
# highest F1 score will be chosen
rows = []
for model_name, results in next_app_all_results.items():
    rows.append({
        'Model': model_name,
        'Accuracy': f"{results['accuracy']*100:.1f}%",
        'Precision': f"{results['precision']*100:.1f}%",
        'Recall': f"{results['recall']*100:.1f}%",
        'F1 Score': f"{results['f1']*100:.1f}%",
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
print(f'\nBest model: {next_app_top_model_name}')

            Model Accuracy Precision Recall F1 Score
     Markov Chain     8.0%      8.6%   8.0%     7.7%
    Decision Tree     8.0%      5.5%   8.0%     6.5%
    Random Forest    14.0%     21.2%  14.0%    10.7%
Gradient Boosting    12.0%      7.2%  12.0%     8.8%
              KNN     8.0%      5.7%   8.0%     5.6%
      Naive Bayes    12.0%      8.0%  12.0%     8.6%

Best model: Random Forest


In [12]:
# this is for model 2: detecting anomalies
# highest recall score will be chosen
rows = []
for model_name, results in anomaly_all_results.items():
    rows.append({
        'Model': model_name,
        'Accuracy': f"{results['accuracy']*100:.1f}%",
        'Precision': f"{results['precision']*100:.1f}%",
        'Recall': f"{results['recall']*100:.1f}%",
        'F1 Score': f"{results['f1']*100:.1f}%",
    })

df_anomaly = pd.DataFrame(rows)
print(df_anomaly.to_string(index=False))
print(f'\nBest model: {best_anomaly_model_name}')

              Model Accuracy Precision Recall F1 Score
      Decision Tree   100.0%    100.0% 100.0%   100.0%
      Random Forest    99.0%      0.0%   0.0%     0.0%
  Gradient Boosting   100.0%    100.0% 100.0%   100.0%
                KNN    99.0%      0.0%   0.0%     0.0%
        Naive Bayes    99.0%      0.0%   0.0%     0.0%
Logistic Regression   100.0%    100.0% 100.0%   100.0%

Best model: Decision Tree


In [13]:
# CONFUSION MATRICES for anomaly detection
for model_name, cm in anomaly_all_confusion_matrices.items():
    print(f'{model_name}:')
    print(cm)
    print()

Decision Tree:
[[99  0]
 [ 0  1]]

Random Forest:
[[99  0]
 [ 1  0]]

Gradient Boosting:
[[99  0]
 [ 0  1]]

KNN:
[[99  0]
 [ 1  0]]

Naive Bayes:
[[99  0]
 [ 1  0]]

Logistic Regression:
[[99  0]
 [ 0  1]]

